# 単語類似度データセットで単語ベクトルを評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_word_vectors/wordsim_eval.ipynb)

NLP クックブック 09-12 ／ 単語ベクトル

単語ベクトルの良し悪しを、人間が付けた単語類似度スコアとの相関で定量的に評価したい。

In [1]:
!pip install -q gensim

In [2]:
import gensim.downloader as api
from gensim.test.utils import datapath

for name in ["glove-wiki-gigaword-50", "glove-wiki-gigaword-100"]:
    wv = api.load(name)
    pearson, spearman, oov = wv.evaluate_word_pairs(datapath("wordsim353.tsv"))
    print(f"{name:26} Spearman = {spearman.statistic:.3f}  Pearson = {pearson.statistic:.3f}  未知語率 = {oov:.1f}%")

glove-wiki-gigaword-50     Spearman = 0.503  Pearson = 0.507  未知語率 = 0.0%
glove-wiki-gigaword-100    Spearman = 0.533  Pearson = 0.548  未知語率 = 0.0%


In [3]:
# 中身を見る: 人間の評価 (0〜10) とモデルのコサイン類似度
import pandas as pd

df = pd.read_csv(datapath("wordsim353.tsv"), sep="\t", comment="#", names=["w1", "w2", "human"])
df["model"] = [wv.similarity(a.lower(), b.lower()) if a.lower() in wv and b.lower() in wv else None for a, b in zip(df.w1, df.w2)]
df.sample(8, random_state=0)

,w1,w2,human,model
6,plane,car,5.77,0.596141
140,precedent,antecedent,6.04,0.284225
299,day,summer,3.94,0.701506
217,board,recommendation,4.47,0.647220
90,coast,forest,3.15,0.410930
226,consumer,energy,4.75,0.573923
114,tiger,zoo,5.87,0.386814
60,drink,car,3.04,0.337204


- 順位相関 (Spearman) を使うのが慣例。値のスケールが違っても、順序が合っていればよいため。
- WordSim-353 は「類似」と「関連」(coffee-cup) が混ざっている。純粋な類似度を測るなら SimLex-999。日本語には JWSAN や日本語単語類似度データセット (JWSD) がある。